We're going to do a visual comparison of the ProtHash and ESMC embeddings as a sanity check. We're expecting that, if ProtHash successfully learned from its ESMC teacher, the two embeddings will be nearly identical in terms of their 2D plots. Now it's not exactly comparing apples to apples when you have to take two high-dimensional embeddings of likely different dimensions and reduce them both down to only two dimensions - but this is just a sanity check.

Let's kick this party off by defining some configuration variables.

In [ ]:
min_sequence_length=1
max_sequence_length=2048
num_samples=500
batch_size=1

teacher_model_name="esmc_300m"

checkpoint_path="checkpoints/checkpoint.pt"

device="cuda"  # Can be "cuda", "mps", etc.

TEACHER_LAYER_ANCHOR_POINTS = {
    "esmc_300m": (7, 15, 22, 29),
    "esmc_600m": (8, 18, 27, 35),
}

anchor_points = TEACHER_LAYER_ANCHOR_POINTS[teacher_model_name]

Then, we'll load the ESM protein sequence tokenizer and the SwissProt dataset.

In [ ]:
from esm.tokenization import EsmSequenceTokenizer

from data import UniRef50

from torch.utils.data import Subset, DataLoader

tokenizer = EsmSequenceTokenizer()

dataset = UniRef50(
    path="./dataset/uniref50.fasta",
    tokenizer=tokenizer,
    min_sequence_length=min_sequence_length,
    max_sequence_length=max_sequence_length,
)

dataset = Subset(dataset, range(num_samples))

dataloader = DataLoader(
    dataset, batch_size=batch_size, shuffle=True, collate_fn=dataset.dataset.collate_pad_right
)

Next we'll load the teacher model, ESMC, from its pretrained weights.

In [ ]:
from esm.models.esmc import ESMC

teacher = ESMC.from_pretrained(teacher_model_name)

teacher = teacher.to(device)

teacher.eval()

print("Teacher model loaded successfully")

Now you've made it this far it's time for some fun. Let's go down and dirty and load one of our ProtHash model checkpoints into memory.

In [ ]:
import torch

from src.prothash.model import ESMCProtHash

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

student = ESMCProtHash(**checkpoint["model_args"])

student.add_sequence_head()

student.load_state_dict(checkpoint["model"])

student.remove_fake_quantized_tensors()

student = student.to(device)

student.eval()

print("Model checkpoint loaded successfully")

You've made it this far there's no turning back. It's literally life or death from here on out. Next we'll be embedding a subset of the SwissProt dataset with both models. I'll know if you turned back from here.

In [ ]:
teacher_embeddings = [[] for _ in range(4)]
student_embeddings = [[] for _ in range(4)]
masks = [[] for _ in range(4)]

for x in dataloader:
    x = x.to(device)

    with torch.no_grad():
        out = teacher.forward(x)

    embeddings = student.embed(x)

    for i in range(4):
        teacher_embeddings[i].append(out.hidden_states[anchor_points[i]].cpu())

    student_embeddings[0].append(embeddings.stage1.cpu())
    student_embeddings[1].append(embeddings.stage2.cpu())
    student_embeddings[2].append(embeddings.stage3.cpu())
    student_embeddings[3].append(embeddings.stage4.cpu())

    mask = (x != tokenizer.pad_token_id).cpu()
    for i in range(4):
        masks[i].append(mask)

max_len = max(t.size(1) for lst in teacher_embeddings for t in lst)


def pad_to_max(tensors, max_len):
    padded = []
    for t in tensors:
        if t.dim() == 3:
            b, l, d = t.shape
            if l < max_len:
                pad = torch.zeros(b, max_len - l, d, dtype=t.dtype)
                t = torch.cat([t, pad], dim=1)
        else:
            b, l = t.shape
            if l < max_len:
                pad = torch.zeros(b, max_len - l, dtype=t.dtype)
                t = torch.cat([t, pad], dim=1)
        padded.append(t)
    return torch.cat(padded, dim=0)


teacher_embeddings = [pad_to_max(lst, max_len) for lst in teacher_embeddings]
student_embeddings = [pad_to_max(lst, max_len) for lst in student_embeddings]
masks = [pad_to_max(lst, max_len) for lst in masks]

Now let's measure the cosine similarity between the teacher's embeddings and the student's embeddings projected to the teacher's vector space.

In [ ]:
from metrics import CosineSimilarity

cosine_similarity_metric = CosineSimilarity()

for i in range(4):
    cosine_similarity_metric.reset()

    cosine_similarity_metric.update(student_embeddings[i], teacher_embeddings[i], masks[i])

    cs = cosine_similarity_metric.compute()
    
    print(f"Anchor {anchor_points[i]} (stage{i+1}): cosine similarity = {cs}")

Here's where things get really crazy. Let's first find the 128 most expressive dimensions of each embedding matrix. Then, let's find a 2-dimensional manifold of that vector space.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

all_s_tsne = []
all_t_tsne = []

for i in range(4):
    pca = PCA(n_components=128)
    tsne = TSNE(n_components=2)

    s = student_embeddings[i].float().numpy()
    t = teacher_embeddings[i].float().numpy()

    s_pca = pca.fit_transform(s)
    t_pca = pca.fit_transform(t)

    s_tsne = tsne.fit_transform(s_pca)
    t_tsne = tsne.fit_transform(t_pca)

    all_s_tsne.append(s_tsne)
    all_t_tsne.append(t_tsne)

Finally let's plot those manifolds so we can attempt to view the differences between the two embeddings in 2 dimensions.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 2, figsize=(12, 24))

for i in range(4):
    axes[i, 0].scatter(
        all_s_tsne[i][:, 0], all_s_tsne[i][:, 1], s=5, alpha=0.7, color="blue"
    )
    axes[i, 0].set_title(f"Student stage{i+1} \u2192 Anchor {anchor_points[i]}")

    axes[i, 1].scatter(
        all_t_tsne[i][:, 0], all_t_tsne[i][:, 1], s=5, alpha=0.7, color="orange"
    )
    axes[i, 1].set_title(f"Teacher layer {anchor_points[i]} \u2192 Anchor {anchor_points[i]}")

    for ax in axes[i]:
        ax.set_xlabel("x")
        ax.set_ylabel("y")

plt.tight_layout()
plt.show()